# 05 - Run Classical Baseline Optimizers (CMA-ES, DE, PSO)

This notebook benchmarks the classical baseline algorithms (**CMA-ES**, **Differential Evolution (DE)**, and **Particle Swarm Optimization (PSO)**) across all experimental conditions ($D \in \{2, 3, 5\}$, $\sigma \in \{0.0, 0.05\}$, $f_i \in \{1, 8, 11, 15, 21\}$) over $N=10$ independent instances.

### Key Principles & Synergy with Notebook 04:
1. **Unified Storage**: Outputs directly to `results/evaluations/{dim}D/std_{noise_std}/f{problem_id}/{algo_name}/`.
2. **Pre-Flight Diagnostics**: Inspects existing data on disk and skips completed/valid evaluations in milliseconds.
3. **Standardized Telemetry**: Attaches `ioh.logger.Analyzer` across instances $1 \dots 10$ and generates `provenance.json`.
4. **Ground-Truth Precision**: Evaluates candidate solutions against clean objectives to prevent noisy ranking bias.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML

# Add src to path
cwd = Path('.').resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from shared.config import RESULTS_DIR
from benchmarking import BenchmarkEvaluationService, BASELINES

# ── User Execution Controls ──────────────────────────────────────────────────
FORCE_REEVALUATE = False            # Set to True to re-run and overwrite existing completed logs
FILTER_BASELINES = None             # e.g., ['pso'] or ['cmaes', 'de'] (None evaluates all)
FILTER_PROBLEMS  = None             # e.g., [1, 8] (None evaluates all)
FILTER_DIMS      = None             # e.g., [2, 3] (None evaluates all)
FILTER_NOISE     = None             # e.g., [0.0, 0.05] (None evaluates all)
N_RUNS           = 10               # Independent runs per condition (instances 1..10)

EVALUATIONS_DIR = RESULTS_DIR / 'evaluations' / 'traces'

service = BenchmarkEvaluationService(eval_dir=EVALUATIONS_DIR, n_runs=N_RUNS)
UNIQUE_CONFIGS = service.sqlite_repo.get_target_conditions()

print(f'🎯 Discovered {len(UNIQUE_CONFIGS)} target experimental conditions.')
print(f'✅ Configured {len(BASELINES)} baseline algorithms: {", ".join(BASELINES.keys())}')

## 1. Pre-Flight Diagnostic Dashboard: Baseline Coverage & Workload Audit

Scans `results/evaluations/` across all conditions to categorize what is already completed vs queued.

In [ ]:
import pandas as pd
from IPython.display import display, HTML

def render_html_dashboard(
    df_audit: pd.DataFrame,
    title: str = "Benchmark Evaluation Pre-Flight Audit",
    subtitle: str = "Real-time status of empirical evaluation runs",
    target_runs: int = 10,
    group_column: str = "model",
) -> str:
    """Render responsive HTML pre-flight dashboard for Jupyter display."""
    if df_audit.empty:
        return "<div>No audit data available.</div>"

    grp_col = (
        group_column
        if group_column in df_audit.columns
        else ("baseline" if "baseline" in df_audit.columns else df_audit.columns[0])
    )

    summary_rows = []
    for name, grp in df_audit.groupby(grp_col):
        total = len(grp)
        completed = len(grp[grp["status"] == "COMPLETED"])
        pending = len(grp[grp["status"] == "PENDING"])
        needs_rerun = len(grp[grp["status"] == "NEEDS_RERUN"])
        missing_code = len(grp[grp["status"] == "MISSING_CODE"])
        to_run_mask = grp["status"].isin(["PENDING", "NEEDS_RERUN"])
        if "is_filtered" in grp.columns:
            to_run_mask = to_run_mask & (~grp["is_filtered"])
        to_run = len(grp[to_run_mask])
        pct = (completed / total * 100) if total > 0 else 0.0
        summary_rows.append({
            "Group": str(name).upper(),
            "Total Tasks": total,
            "Completed": completed,
            "Pending": pending,
            "Needs Rerun": needs_rerun,
            "Missing Code": missing_code,
            "Queue to Run": to_run,
            "Progress (%)": pct,
        })

    df_summary = pd.DataFrame(summary_rows)
    total_queue = int(df_summary["Queue to Run"].sum())
    total_completed = int(df_summary["Completed"].sum())
    total_tasks = int(df_summary["Total Tasks"].sum())
    overall_pct = (total_completed / total_tasks * 100) if total_tasks > 0 else 0.0

    html = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 950px; margin: 15px 0;">
    <div style="display: flex; align-items: center; justify-content: space-between; margin-bottom: 16px; border-bottom: 2px solid #E2E8F0; padding-bottom: 8px;">
        <div>
            <h2 style="margin: 0; color: #0F172A; font-size: 20px; font-weight: 700; display: flex; align-items: center; gap: 8px;">
                🚀 {title}
            </h2>
            <p style="margin: 4px 0 0 0; color: #64748B; font-size: 13px;">{subtitle}</p>
        </div>
        <div style="background: #EEF2F6; padding: 6px 12px; border-radius: 20px; font-size: 12px; font-weight: 600; color: #334155;">
            Target: N={target_runs} runs / condition
        </div>
    </div>

    <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin-bottom: 20px;">
        <div style="background: linear-gradient(135deg, #1E293B 0%, #0F172A 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #94A3B8;">Total Targets</div>
            <div style="font-size: 24px; font-weight: 700; color: #F8FAFC; margin-top: 4px;">{total_tasks}</div>
            <div style="font-size: 11px; color: #64748B; margin-top: 2px;">Across {len(df_summary)} Categories</div>
        </div>
        <div style="background: linear-gradient(135deg, #065F46 0%, #047857 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #A7F3D0;">Completed & Valid</div>
            <div style="font-size: 24px; font-weight: 700; color: #ECFDF5; margin-top: 4px;">{total_completed}</div>
            <div style="font-size: 11px; color: #D1FAE5; margin-top: 2px;">{overall_pct:.1f}% Overall Progress</div>
        </div>
        <div style="background: linear-gradient(135deg, #C2410C 0%, #9A3412 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #FED7AA;">Queue to Run</div>
            <div style="font-size: 24px; font-weight: 700; color: #FFF7ED; margin-top: 4px;">{total_queue}</div>
            <div style="font-size: 11px; color: #FFEDD5; margin-top: 2px;">{total_queue * target_runs} Total Runs</div>
        </div>
        <div style="background: linear-gradient(135deg, #4338CA 0%, #3730A3 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #C7D2FE;">Est. Runtime</div>
            <div style="font-size: 24px; font-weight: 700; color: #EEF2FF; margin-top: 4px;">~{total_queue * 1.2:.0f}m</div>
            <div style="font-size: 11px; color: #E0E7FF; margin-top: 2px;">@ ~1.2s per run</div>
        </div>
    </div>

    <div style="background: #F8FAFC; border: 1px solid #E2E8F0; border-radius: 10px; padding: 16px; margin-bottom: 8px;">
        <div style="font-size: 13px; font-weight: 700; color: #1E293B; margin-bottom: 12px; text-transform: uppercase; letter-spacing: 0.04em;">
            Completion Progress Breakdown
        </div>
"""
    for _, row in df_summary.iterrows():
        grp_name = row["Group"]
        tot = int(row["Total Tasks"])
        comp = int(row["Completed"])
        pend = int(row["Pending"])
        rerun = int(row["Needs Rerun"])
        q = int(row["Queue to Run"])
        pct = float(row["Progress (%)"])
        color = "#10B981" if pct > 75 else ("#3B82F6" if pct > 25 else "#F59E0B")
        rerun_pct = (rerun / tot * 100) if tot > 0 else 0

        html += f"""
        <div style="margin-bottom: 14px;">
            <div style="display: flex; justify-content: space-between; font-size: 13px; font-weight: 600; color: #334155; margin-bottom: 4px;">
                <span>⚙️ <strong style="color: #0F172A;">{grp_name}</strong> &nbsp;({comp}/{tot} Completed)</span>
                <span style="color: {color}; font-weight: 700;">{pct:.1f}%</span>
            </div>
            <div style="background: #E2E8F0; border-radius: 6px; height: 10px; overflow: hidden; display: flex;">
                <div style="background: #10B981; width: {pct}%; transition: width 0.3s;"></div>
                <div style="background: #EF4444; width: {rerun_pct}%;"></div>
            </div>
            <div style="display: flex; gap: 14px; font-size: 11px; color: #64748B; margin-top: 5px;">
                <span>✅ Completed: <strong style="color: #059669;">{comp}</strong></span>
                <span>⏳ Pending: <strong style="color: #D97706;">{pend}</strong></span>
                <span>⚠️ Needs Rerun: <strong style="color: #DC2626;">{rerun}</strong></span>
                <span>🎯 Queue: <strong style="color: #2563EB;">{q}</strong></span>
            </div>
        </div>
"""
    html += """
    </div>
</div>
"""
    return html

# Audit baseline completion and validity status via BenchmarkEvaluationService
df_audit_base = service.audit_baselines_workload(
    conditions=UNIQUE_CONFIGS,
    baselines=FILTER_BASELINES,
    filter_dims=FILTER_DIMS,
    filter_noise=FILTER_NOISE,
    filter_problems=FILTER_PROBLEMS,
)

# Render styled pre-flight visual dashboard
html_dash = render_html_dashboard(
    df_audit_base,
    title="Baseline Evaluation Pre-Flight Audit",
    target_runs=N_RUNS,
    group_column="baseline",
)
display(HTML(html_dash))


## 2. Execute Baseline Benchmarks (N=10 Independent Runs)

Runs all queued classical optimizers with `ioh.logger.Analyzer` and writes `provenance.json`.

In [ ]:
# Run all pending baseline evaluations via BenchmarkEvaluationService
results_base_df = service.run_baselines(
    conditions=UNIQUE_CONFIGS,
    baselines=FILTER_BASELINES,
    filter_dims=FILTER_DIMS,
    filter_noise=FILTER_NOISE,
    filter_problems=FILTER_PROBLEMS,
    force_rerun=FORCE_REEVALUATE,
)